# 🔬 Benchmark y validación interna del dataset sintético EBLET

## 🎯 Objetivo

Este notebook presenta la **validación estadística interna** del benchmark sintético desarrollado para el framework EBLET. Su finalidad es comprobar que los datos generados presentan una estructura coherente, consistente y diferenciada entre los escenarios organizacionales definidos durante el proceso de construcción del benchmark.

## 📋 Benchmark analizado

El proceso de validación se realiza sobre un conjunto de datos compuesto por:

- **250 empresas sintéticas**
- **12.500 empleados**
- **5 escenarios organizacionales**
- **67 ítems Likert por empleado**
- **Indicadores derivados de burnout, boreout, bienestar, intención de rotación y cultura organizacional (CVF)**

## 🔍 Alcance de la validación

A diferencia del notebook anterior, centrado en la metodología de construcción del benchmark, este documento evalúa la calidad estadística del conjunto de datos generado mediante los siguientes análisis:

1. Estadísticos descriptivos de los principales indicadores.
2. Diferencias entre escenarios organizacionales.
3. Consistencia interna de las dimensiones mediante el alfa de Cronbach.
4. Relaciones entre indicadores mediante matrices de correlación.
5. Estructura latente de las respuestas mediante Análisis de Componentes Principales (PCA).

Los resultados obtenidos constituyen una **validación interna del benchmark sintético**, verificando que reproduce patrones coherentes con la literatura sobre bienestar laboral y comportamiento organizacional. No obstante, estos análisis **no sustituyen una validación psicométrica externa**, que requeriría la aplicación del instrumento EBLET a una muestra real de trabajadores.

# Resumen del benchmark

El benchmark sintético generado para el framework EBLET constituye la base de referencia utilizada para interpretar los resultados obtenidos por las organizaciones evaluadas. El conjunto de datos se diseñó para representar cinco escenarios organizacionales diferenciados, manteniendo un equilibrio en el número de empresas y empleados por escenario.

En total, el benchmark está compuesto por 250 empresas y 12.500 empleados, distribuidos homogéneamente entre los cinco escenarios definidos. Cada empleado dispone de respuestas simuladas a los 67 ítems Likert de la encuesta EBLET, así como de los indicadores psicológicos, organizacionales y económicos derivados del proceso de simulación.

In [1]:
# ============================================================
# IMPORTS Y CONFIGURACIÓN
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# Localizar automáticamente la raíz del proyecto.
# El notebook debe estar dentro de la carpeta notebooks/.
RAIZ_PROYECTO = Path.cwd()

if RAIZ_PROYECTO.name.lower() == "notebooks":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

SRC_PATH = RAIZ_PROYECTO / "src"
DATASETS_PATH = RAIZ_PROYECTO / "datasets"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


from config import (
    N_PREGUNTAS_TOTAL,
    preguntas_dimension,
    preguntas_cvf,
)

from estilos import (
    aplicar_estilo,
    COLORES_CULTURA,
)


ESCENARIOS = [
    "saludable",
    "estable",
    "riesgo_boreout",
    "riesgo_burnout",
    "critico",
]

ETIQUETAS_ES = {
    "saludable": "Saludable",
    "estable": "Estable",
    "riesgo_boreout": "Riesgo Boreout",
    "riesgo_burnout": "Riesgo Burnout",
    "critico": "Crítico",
}

COLORES_ESCENARIOS = {
    "saludable": "#2E8B57",
    "estable": "#4C78A8",
    "riesgo_boreout": "#F2C14E",
    "riesgo_burnout": "#E67E22",
    "critico": "#C0392B",
}



📁 Directorio actual: C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python\notebooks
📁 Ruta real de datasets: C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python\notebooks\datasets


## 📊 Resumen del Dataset

In [2]:
# ============================================================
# 1. CARGA Y CONTROL DE CALIDAD DEL BENCHMARK
# ============================================================

datasets = []

for escenario in ESCENARIOS:
    ruta = DATASETS_PATH / escenario / "empleados.csv"

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró el dataset del escenario "
            f"{escenario!r}: {ruta}"
        )

    df_escenario = pd.read_csv(ruta)
    df_escenario["escenario"] = escenario
    datasets.append(df_escenario)

df_benchmark = pd.concat(
    datasets,
    ignore_index=True,
)


# Identificar dinámicamente las preguntas disponibles.
preguntas_cols = [
    f"q{i}"
    for i in range(1, N_PREGUNTAS_TOTAL + 1)
]

preguntas_faltantes = [
    columna
    for columna in preguntas_cols
    if columna not in df_benchmark.columns
]

if preguntas_faltantes:
    raise ValueError(
        "Faltan preguntas en el benchmark: "
        + ", ".join(preguntas_faltantes)
    )


print("📊 RESUMEN DEL BENCHMARK EBLET")
print(f"👥 Empleados: {len(df_benchmark):,}")
print(
    f"🏢 Empresas: "
    f"{df_benchmark['empresa_id'].nunique():,}"
)
print(f"📋 Preguntas Likert: {len(preguntas_cols)}")
print(f"🎯 Escenarios: {df_benchmark['escenario'].nunique()}")

print("\nDistribución por escenario:")

resumen_escenarios = (
    df_benchmark
    .groupby("escenario")
    .agg(
        empleados=("empleado_id", "count"),
        empresas=("empresa_id", "nunique"),
    )
    .reindex(ESCENARIOS)
)

resumen_escenarios.index = [
    ETIQUETAS_ES[escenario]
    for escenario in resumen_escenarios.index
]

print(resumen_escenarios.to_string())

print("\nControl de calidad:")
print(
    "   IDs de empleados únicos:",
    df_benchmark["empleado_id"].is_unique,
)
print(
    "   IDs de empresas:",
    df_benchmark["empresa_id"].nunique(),
)
print(
    "   Valores Likert fuera de 1-5:",
    int(
        (
            (df_benchmark[preguntas_cols] < 1)
            | (df_benchmark[preguntas_cols] > 5)
        ).sum().sum()
    ),
)
print(
    "   Respuestas ausentes:",
    int(df_benchmark[preguntas_cols].isna().sum().sum()),
)

📊 RESUMEN DEL BENCHMARK EBLET
👥 Empleados: 12,500
🏢 Empresas: 250
📋 Preguntas Likert: 67
🎯 Escenarios: 5

Distribución por escenario:
                empleados  empresas
Saludable            2500        50
Estable              2500        50
Riesgo Boreout       2500        50
Riesgo Burnout       2500        50
Crítico              2500        50

Control de calidad:
   IDs de empleados únicos: True
   IDs de empresas: 250
   Valores Likert fuera de 1-5: 0
   Respuestas ausentes: 0


# 1. Estadísticos descriptivos de los KPIs

Como primer paso de la validación se analizan los principales indicadores calculados a partir de las respuestas de la encuesta. Este análisis permite comprobar que cada uno de los escenarios organizacionales presenta valores coherentes con el modelo teórico utilizado para generar el benchmark.

Los indicadores evaluados son burnout, boreout, bienestar e intención de rotación, calculados mediante la agregación de las dimensiones correspondientes de los instrumentos psicométricos integrados en EBLET.

## 2. Distribución de los KPIs por escenario

La distribución de los principales indicadores permite comprobar si los escenarios organizacionales definidos durante la construcción del benchmark presentan perfiles diferenciados y coherentes con su formulación teórica.

Los diagramas de caja muestran la mediana, la dispersión y el rango de las puntuaciones individuales de burnout, boreout, bienestar, intención de cambio y contexto organizacional.

![Distribución de KPIs por escenario organizacional](../../visualizaciones/06_distribucion_kpis_escenario.png)

**Figura 1.** Distribución de los principales KPIs según el escenario organizacional. Elaboración propia.

Los resultados muestran una separación clara entre los perfiles organizacionales simulados. El escenario saludable presenta valores reducidos de burnout, boreout e intención de cambio, acompañados de niveles elevados de bienestar y contexto organizacional.

El escenario de riesgo de burnout se caracteriza por puntuaciones elevadas de agotamiento y una disminución del bienestar. Por su parte, el escenario de riesgo de boreout presenta niveles especialmente altos de aburrimiento e infraocupación, aunque el burnout permanece en niveles reducidos.

Finalmente, el escenario crítico combina valores elevados de burnout, boreout e intención de cambio con puntuaciones bajas de bienestar y contexto organizacional. Esta diferenciación respalda la coherencia interna del proceso de generación sintética y confirma que los escenarios representan perfiles organizacionales suficientemente distintos.

## 3. Correlaciones entre los principales KPIs

El análisis de correlaciones permite examinar las relaciones existentes entre los principales indicadores del benchmark sintético. Esta comprobación resulta relevante para determinar si los KPIs presentan asociaciones coherentes con el modelo conceptual utilizado durante la generación de los datos.

La matriz muestra el coeficiente de correlación de Pearson entre burnout, boreout, bienestar, intención de cambio y contexto organizacional. Los valores próximos a 1 indican relaciones positivas, los próximos a -1 reflejan relaciones negativas y los cercanos a 0 señalan una asociación lineal débil.

![Matriz de correlaciones entre los principales KPIs](../../visualizaciones/07_correlaciones_kpis.png)

**Figura 2.** Matriz de correlaciones de Pearson entre los principales KPIs del benchmark sintético. Elaboración propia.

La matriz refleja relaciones coherentes con la lógica teórica del benchmark. El burnout y el boreout presentan asociaciones negativas con el bienestar, lo que indica que el incremento de los riesgos psicosociales se relaciona con una peor experiencia laboral.

La intención de cambio muestra una relación positiva con los indicadores de burnout y boreout. Este resultado sugiere que los empleados con mayores niveles de agotamiento, desmotivación o infraocupación presentan también una mayor predisposición a abandonar la organización.

Por otra parte, el contexto organizacional mantiene una relación positiva con el bienestar y negativa con los indicadores de riesgo y la intención de cambio. Este patrón respalda la idea de que unas condiciones organizacionales favorables actúan como factor protector frente al deterioro psicológico y la rotación potencial.

En conjunto, las correlaciones obtenidas respaldan la coherencia interna del benchmark, ya que los indicadores se relacionan en las direcciones esperadas según el modelo conceptual de EBLET.

## 4. Comparación de medias entre escenarios (ANOVA)"

Con el objetivo de comprobar si los cinco escenarios organizacionales presentan diferencias estadísticamente significativas en los principales indicadores del benchmark, se realizó un análisis de la varianza (ANOVA) de un factor para cada uno de los KPIs.

Este contraste permite determinar si las diferencias observadas entre las medias de los escenarios son superiores a las esperadas por el efecto del azar. La Tabla 1 resume los resultados obtenidos.

In [28]:
from scipy import stats

resultados_anova = []

for kpi in kpi_cols:

    grupos = [
        grupo[kpi].values
        for _, grupo in df_benchmark.groupby("escenario")
    ]

    F, p = stats.f_oneway(*grupos)

    resultados_anova.append({
        "KPI": NOMBRES_KPI[kpi],
        "F": round(F, 2),
        "p-valor": "<0.001" if p < 0.001 else f"{p:.3f}",
        "Resultado": (
            "Significativo"
            if p < 0.05
            else "No significativo"
        )
    })

tabla_anova = pd.DataFrame(resultados_anova)

display(tabla_anova)

,KPI,F,p-valor,Resultado
0,Burnout,12037.06,<0.001,Significativo
1,Boreout,10976.38,<0.001,Significativo
2,Bienestar,7525.39,<0.001,Significativo
3,Intención de cambio,7152.99,<0.001,Significativo
4,Contexto,11816.07,<0.001,Significativo


**Tabla 1.** Resultados del análisis de la varianza (ANOVA) para los principales indicadores del benchmark sintético. Elaboración propia.

Los resultados muestran diferencias estadísticamente significativas entre los cinco escenarios organizacionales para todos los indicadores analizados (p < 0,001). Este resultado confirma que las variaciones observadas en burnout, boreout, bienestar, intención de cambio y contexto organizacional no pueden atribuirse al azar, sino que reflejan diferencias sistemáticas entre los perfiles organizacionales definidos durante la construcción del benchmark.


## 5. Consistencia interna de las dimensiones (alfa de Cronbach)

Con el fin de evaluar la consistencia interna de las dimensiones incluidas en el benchmark sintético, se calculó el coeficiente alfa de Cronbach para cada una de las escalas que componen el cuestionario EBLET.

Este coeficiente permite estimar el grado de homogeneidad entre los ítems que integran una misma dimensión. De forma general, valores superiores a 0,70 se consideran aceptables, mientras que valores por encima de 0,80 indican una buena consistencia interna.

In [ ]:
# ============================================================
# ALFA DE CRONBACH - FIABILIDAD DE LAS DIMENSIONES
# ============================================================

from scores import analisis_fiabilidad

# Calcular la fiabilidad del benchmark
fiabilidad = analisis_fiabilidad(df_benchmark)

# Clasificación de la consistencia interna
def interpretar_alpha(alpha):
    if alpha >= 0.90:
        return "Excelente"
    elif alpha >= 0.80:
        return "Buena"
    elif alpha >= 0.70:
        return "Aceptable"
    else:
        return "Insuficiente"

fiabilidad["Interpretación"] = (
    fiabilidad["Alfa_Cronbach"]
    .apply(interpretar_alpha)
)

fiabilidad = fiabilidad.rename(columns={
    "Dimensión": "Dimensión",
    "N_ítems": "N.º de ítems",
    "Alfa_Cronbach": "α de Cronbach"
})

display(fiabilidad)

,Dimensión,N.º de ítems,α de Cronbach,Interpretación
0,Burnout - Agotamiento (MBI-GS),7,0.984,Excelente
1,Burnout - Cinismo (MBI-GS),7,0.978,Excelente
2,Burnout - Ineficacia (MBI-GS),7,0.984,Excelente
3,Aburrimiento Laboral (EAL),8,0.984,Excelente
4,Bienestar (WHO-5),5,0.977,Excelente
5,Satisfacción Laboral,4,0.963,Excelente
6,Autoeficacia (Bandura),3,0.923,Excelente
7,Intención de Rotación (Mobley),3,0.935,Excelente
8,Contexto Organizacional (JD-R),15,0.984,Excelente
9,Cultura CVF - Adhocracia,2,0.710,Aceptable


Tabla 2. Consistencia interna de las principales dimensiones del cuestionario EBLET medida mediante el coeficiente alfa de Cronbach. Elaboración propia.

Los resultados muestran que todas las dimensiones presentan valores del coeficiente alfa de Cronbach superiores al umbral de 0,70, lo que indica una adecuada consistencia interna de las escalas utilizadas. En particular, las dimensiones con valores superiores a 0,80 evidencian una buena fiabilidad, mientras que aquellas por encima de 0,90 alcanzan un nivel excelente. En conjunto, estos resultados respaldan la coherencia interna del instrumento y constituyen una evidencia adicional de la calidad del benchmark sintético.

## 6. Perfil medio de los escenarios organizacionales

Con el fin de sintetizar las diferencias entre los cinco escenarios definidos durante la construcción del benchmark, se representó el perfil medio de los principales indicadores organizacionales. Cada línea muestra la puntuación media obtenida por un escenario en los cinco KPIs analizados.

![Perfil medio de los escenarios organizacionales](../../visualizaciones/09_perfil_escenarios.png)

**Figura 3.** Perfil medio de los principales KPIs en cada escenario organizacional. Elaboración propia.

La representación conjunta de los indicadores permite apreciar la identidad de cada uno de los escenarios organizacionales. El escenario saludable presenta un contexto organizacional favorable, elevados niveles de bienestar y bajas puntuaciones en burnout, boreout e intención de cambio.

Por el contrario, el escenario crítico concentra los valores más desfavorables en prácticamente todos los indicadores, reflejando simultáneamente un contexto organizacional deteriorado, un menor bienestar y mayores niveles de riesgos psicosociales y rotación potencial.

Los escenarios de riesgo de burnout y riesgo de boreout muestran perfiles claramente diferenciados, con incrementos específicos en el indicador que les da nombre, mientras que el escenario estable ocupa una posición intermedia en la mayoría de los KPIs. En conjunto, estos resultados evidencian que el benchmark reproduce perfiles organizacionales coherentes y fácilmente distinguibles entre sí.

## 7. 🧬 PCA - Análisis de Componentes Principales

**Pregunta**: ¿Las preguntas se agrupan de forma lógica según las dimensiones teóricas?

**Objetivo**: Verificar que la estructura latente de los datos coincide con la estructura teórica de la encuesta.


- Las preguntas de burnout (q16-q36) deberían mostrar patrones comunes.
- Las preguntas de boreout (q37-q44) deberían agruparse.
- Las preguntas de bienestar (q45-q56) deberían mostrar una estructura relacionada.
- Las preguntas de intención de cambio (q57-q59) deberían presentar un patrón propio.
- Las preguntas de cultura CVF (q60-q67) deberían aparecer en componentes diferenciados.

In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Columnas correspondientes a los ítems del cuestionario

columnas_items = sorted(
    [
        columna
        for columna in df_benchmark.columns
        if columna.startswith("q")
        and columna[1:].isdigit()
    ],
    key=lambda columna: int(columna[1:])
)

print(
    f"Se han seleccionado {len(columnas_items)} ítems:"
)
print(columnas_items)

# Selección de los ítems utilizados en el PCA
X_pca = df_benchmark[columnas_items].dropna()

# Estandarización
X_pca_escalado = StandardScaler().fit_transform(X_pca)

# Ajuste del PCA
pca = PCA()
componentes = pca.fit_transform(X_pca_escalado)

# Varianza explicada
varianza = pca.explained_variance_ratio_
varianza_acumulada = varianza.cumsum()

Se han seleccionado 67 ítems:
['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22', 'q23', 'q24', 'q25', 'q26', 'q27', 'q28', 'q29', 'q30', 'q31', 'q32', 'q33', 'q34', 'q35', 'q36', 'q37', 'q38', 'q39', 'q40', 'q41', 'q42', 'q43', 'q44', 'q45', 'q46', 'q47', 'q48', 'q49', 'q50', 'q51', 'q52', 'q53', 'q54', 'q55', 'q56', 'q57', 'q58', 'q59', 'q60', 'q61', 'q62', 'q63', 'q64', 'q65', 'q66', 'q67']


## 7. Estructura latente de los datos mediante PCA

Con el objetivo de explorar la estructura subyacente de las respuestas del cuestionario, se realizó un Análisis de Componentes Principales (PCA) sobre los ítems incluidos en el benchmark.

Antes de aplicar el análisis, las variables fueron estandarizadas para evitar que las diferencias de escala o dispersión condicionaran el peso de cada ítem. La figura muestra la proporción de varianza explicada por cada componente principal y la varianza acumulada alcanzada al incorporar sucesivamente nuevos componentes.

![Varianza explicada por los componentes principales](../../visualizaciones/10_varianza_explicada_pca.png)

**Figura 4.** Varianza individual y acumulada explicada por los diez primeros componentes principales. La línea discontinua indica el umbral de referencia del 80 %. Elaboración propia.

In [37]:
n_componentes_80 = (
    next(
        i + 1
        for i, valor in enumerate(varianza_acumulada)
        if valor >= 0.80
    )
)

print(
    f"Número de componentes necesarios para explicar "
    f"al menos el 80 % de la varianza: {n_componentes_80}"
)

Número de componentes necesarios para explicar al menos el 80 % de la varianza: 5


Los primeros componentes concentran una parte relevante de la variabilidad total de las respuestas, mientras que la contribución individual disminuye progresivamente a medida que se incorporan nuevos componentes.

El umbral del 80 % de varianza acumulada se alcanza con **5 componentes principales**. Este resultado indica que la información contenida en los ítems puede resumirse en un número de dimensiones inferior al número total de variables originales, aunque la estructura no se concentra necesariamente en uno o dos únicos factores.

La reducción progresiva de la varianza explicada por cada componente es consistente con la naturaleza multidimensional del cuestionario EBLET, que integra constructos diferentes como contexto organizacional, bienestar, burnout, boreout, intención de cambio y cultura organizacional.

### Cargas de los componentes principales

Para facilitar la interpretación de la estructura obtenida mediante el PCA, se analizaron las cargas de los ítems en los cinco primeros componentes principales.

Las cargas representan la relación entre cada ítem original y el componente correspondiente. Cuanto mayor es su valor absoluto, mayor es la contribución del ítem a la definición del componente. El signo indica la dirección de la relación, pero no su importancia relativa.

La siguiente tabla recoge los diez ítems con mayor carga absoluta en cada uno de los cinco primeros componentes.

In [ ]:
# ============================================================
# CARGAS DE LOS COMPONENTES PRINCIPALES
# ============================================================

def obtener_dimension_pregunta(pregunta):
    numero = int(pregunta.replace("q", ""))

    if 1 <= numero <= 15:
        return "Contexto"
    elif 16 <= numero <= 22:
        return "Burnout - Agotamiento"
    elif 23 <= numero <= 29:
        return "Burnout - Cinismo"
    elif 30 <= numero <= 36:
        return "Burnout - Eficacia"
    elif 37 <= numero <= 44:
        return "Boreout"
    elif 45 <= numero <= 49:
        return "WHO-5"
    elif 50 <= numero <= 53:
        return "Satisfacción"
    elif 54 <= numero <= 56:
        return "Autoeficacia"
    elif 57 <= numero <= 59:
        return "Intención de cambio"
    elif 60 <= numero <= 61:
        return "CVF - Adhocracia"
    elif 62 <= numero <= 63:
        return "CVF - Clan"
    elif 64 <= numero <= 65:
        return "CVF - Mercado"
    else:
        return "CVF - Jerárquica"


# Matriz de cargas de los componentes
cargas = pd.DataFrame(
    pca.components_.T,
    columns=[
        f"PC{i + 1}"
        for i in range(pca.n_components_)
    ],
    index=columnas_items,
)


# Seleccionar las diez cargas más altas
# en valor absoluto para los cinco primeros componentes
resultados_cargas = []

for i in range(5):
    pc = f"PC{i + 1}"

    top_preguntas = (
        cargas[pc]
        .abs()
        .sort_values(ascending=False)
        .head(10)
        .index
    )

    for orden, pregunta in enumerate(
        top_preguntas,
        start=1
    ):
        resultados_cargas.append({
            "Componente": pc,
            "Varianza explicada (%)": round(
                varianza[i] * 100,
                2
            ),
            "Orden": orden,
            "Ítem": pregunta,
            "Dimensión": obtener_dimension_pregunta(
                pregunta
            ),
            "Carga": round(
                cargas.loc[pregunta, pc],
                3
            ),
            "Carga absoluta": round(
                abs(cargas.loc[pregunta, pc]),
                3
            ),
        })


tabla_cargas = pd.DataFrame(resultados_cargas)

display(
    tabla_cargas[
        [
            "Componente",
            "Varianza explicada (%)",
            "Orden",
            "Ítem",
            "Dimensión",
            "Carga",
        ]
    ]
)

,Componente,Varianza explicada (%),Orden,Ítem,Dimensión,Carga
0,PC1,55.88,1,q2,Contexto,0.142
1,PC1,55.88,2,q6,Contexto,0.142
2,PC1,55.88,3,q4,Contexto,0.142
3,PC1,55.88,4,q7,Contexto,0.142
4,PC1,55.88,5,q10,Contexto,0.142
5,PC1,55.88,6,q1,Contexto,0.142
6,PC1,55.88,7,q12,Contexto,0.142
7,PC1,55.88,8,q5,Contexto,0.142
8,PC1,55.88,9,q11,Contexto,0.142
9,PC1,55.88,10,q3,Contexto,0.142


**Tabla 3.** Ítems con mayor carga absoluta en los cinco primeros componentes principales. La varianza explicada corresponde a la contribución individual de cada componente. Elaboración propia.

El análisis de las cargas permite identificar qué dimensiones del cuestionario contribuyen en mayor medida a cada componente principal. La presencia de varios ítems pertenecientes a una misma dimensión dentro de un componente sugiere que este recoge un patrón común asociado a dicho constructo.

No obstante, los componentes principales no deben interpretarse automáticamente como factores psicológicos independientes. El PCA es una técnica de reducción de dimensionalidad orientada a resumir la variabilidad de los datos, por lo que la interpretación sustantiva debe realizarse considerando conjuntamente las cargas, la varianza explicada y el contenido teórico de los ítems.

### Interpretación de las cargas

Las cargas muestran la contribución de cada pregunta a los componentes
principales.

- Una carga positiva elevada indica una relación directa con el componente.
- Una carga negativa elevada indica una relación inversa.
- El signo puede invertirse sin alterar el significado matemático del PCA.
- La agrupación de preguntas de una misma dimensión permite comprobar si el
  benchmark conserva la estructura introducida por el modelo generativo.

Estas cargas no deben interpretarse como cargas factoriales confirmatorias.

### Proyección de los empleados en los dos primeros componentes

La proyección bidimensional permite examinar cómo se distribuyen los empleados en el espacio definido por los dos primeros componentes principales. Cada punto representa a un empleado, mientras que los rombos señalan el centroide de cada escenario organizacional.

![Proyección PCA de empleados por escenario](../../visualizaciones/11_proyeccion_pca_escenarios.png)

**Figura 5.** Proyección de los empleados en los dos primeros componentes principales y centroides de los escenarios organizacionales. Elaboración propia.

La posición de los centroides permite observar si los escenarios ocupan regiones diferenciadas dentro del espacio reducido generado por el PCA. Una separación clara entre ellos indicaría que los patrones de respuesta de los empleados varían de forma sistemática según el escenario organizacional.

No obstante, la posible superposición entre observaciones individuales es esperable, ya que los dos primeros componentes representan únicamente una parte de la varianza total. Por tanto, la figura debe interpretarse como una representación simplificada de una estructura multidimensional más compleja.

# Conclusiones

El benchmark EBLET presenta una estructura interna coherente con el diseño generativo empleado. Los cinco escenarios muestran perfiles diferenciados en burnout, boreout, bienestar, contexto e intención de cambio. El ANOVA a nivel empresa confirma diferencias amplias entre escenarios, mientras que los coeficientes de consistencia interna y el PCA reflejan una organización de los ítems compatible con las dimensiones simuladas.

Estos resultados constituyen una validación interna del benchmark sintético y permiten utilizarlo como entorno de desarrollo, comparación y prueba del framework EBLET. No equivalen a una validación psicométrica externa, que requeriría datos procedentes de organizaciones reales.